# 2 — The Full Translation Pipeline: ToolX → DEIX → ToolY

## The Architecture

```
SysML Interior    SysML      SysML-to-DEIX    DEIX         DEIX-to-Sim     Sim        Sim Interior
(sysml: vocab)  → Surface  → Portal         → Interior   → Portal       → Surface  → (sim: vocab)
                  (SHACL)    (CONSTRUCT)      (deix: vocab) (CONSTRUCT)    (SHACL)
```

Each arrow is a governed step:
- **Surface** = SHACL shapes declaring what the target accepts
- **Portal** = SPARQL CONSTRUCT translating between vocabularies
- **Interior** = the named graph holding the data in its native form

DEIX serves as the **canonical intermediate holon**.  Data enters it
from any tool and exits it toward any other tool.  The N×M integration
problem becomes N+M portal definitions.

This notebook builds all three holons, creates the portals, traverses
them, validates at every boundary, and records provenance — using the
actual `holonic` library objects.

In [1]:
from holonic import (
    Holon, TransformPortal, Holarchy,
    validate_membrane, MembraneHealth,
    ProvenanceTracker,
    discover_target_shape, describe_surface,
)
from rdflib import Graph, URIRef, RDF

## Step 1: Build the SysML Tool Holon

The interior uses `sysml:` vocabulary — this is the tool's native
representation, NOT DEIX.

In [3]:
sysml = Holon(
    iri="urn:holon:tool:sysml",
    label="SysML v2 Modeler",
    depth=1,

    interior_ttl="""
        @prefix sysml: <urn:sysml:> .

        <urn:sysml:block:tms> a sysml:Block ;
            sysml:name       "ThermalMgmtSubsystem" ;
            sysml:stereotype "Block" ;
            sysml:package    <urn:sysml:pkg:vehicle> .

        <urn:sysml:param:tms:mass> a sysml:ValueProperty ;
            sysml:name       "mass" ;
            sysml:value      142.3 ;
            sysml:unit       "kg" ;
            sysml:owner      <urn:sysml:block:tms> .

        <urn:sysml:param:tms:power> a sysml:ValueProperty ;
            sysml:name       "maxPowerDraw" ;
            sysml:value      2.8 ;
            sysml:unit       "kW" ;
            sysml:owner      <urn:sysml:block:tms> .

        <urn:sysml:param:tms:flow> a sysml:ValueProperty ;
            sysml:name       "coolantFlowRate" ;
            sysml:value      15.0 ;
            sysml:unit       "L/min" ;
            sysml:owner      <urn:sysml:block:tms> .
    """,

    boundary_ttl="""
        @prefix sysml: <urn:sysml:> .
        <urn:shapes:sysml:BlockShape> a sh:NodeShape ;
            sh:targetClass sysml:Block ;
            sh:property [
                sh:path sysml:name ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation
            ] .
        <urn:shapes:sysml:ValuePropertyShape> a sh:NodeShape ;
            sh:targetClass sysml:ValueProperty ;
            sh:property [
                sh:path sysml:value ;
                sh:minCount 1 ;
                sh:severity sh:Violation
            ] .
    """,

    context_ttl="""
        <urn:holon:tool:sysml> cga:memberOf <urn:holon:domain:engineering> .
    """,
)

print(sysml)
print(f"\n  Interior graph IRI: {sysml.interior.identifier}")
print(f"  Interior vocabulary: sysml: (mock of Cameo)")

Holon('SysML v2 Modeler', depth=1, interior=26t, boundary=13t, projection=0t, context=1t)

  Interior graph IRI: urn:holon:tool:sysml/interior
  Interior vocabulary: sysml: (mock of Cameo)


## Step 2: Build the DEIX Canonical Holon

The DEIX holon's **boundary** defines what valid DEIX data looks like.
This is the self-describing surface that any tool can query to discover
what DEIX accepts.

The DEIX boundary shapes use actual DEIX ontology classes:
- `deix:Digital_Artifact` — the canonical class for tool outputs
- `deix:Digital_Information` — typed parameter values
- Properties from the DEIX ontology namespace

In [4]:
deix = Holon(
    iri="urn:holon:deix:canonical",
    label="DEIX Canonical",
    depth=0,

    # Interior starts EMPTY — it gets populated by portal projections
    interior_ttl="",

    # Boundary: SHACL shapes using DEIX vocabulary.
    # THIS IS THE SELF-DESCRIBING SURFACE.
    # Any tool can query these shapes to discover what DEIX accepts.
    boundary_ttl="""
        @prefix deix: <https://semantic.incose.org/DEIX_Ontology#> .

        # --- What a valid Digital Artifact looks like in DEIX ---
        <urn:shapes:deix:DigitalArtifactShape> a sh:NodeShape ;
            sh:targetClass deix:Digital_Artifact ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Digital Artifact must have a label (DEIX)."
            ] ;
            sh:property [
                sh:path deix:hasParameter ;
                sh:minCount 1 ;
                sh:severity sh:Warning ;
                sh:message "Digital Artifact should declare parameters."
            ] .

        # --- What a valid parameter observation looks like ---
        <urn:shapes:deix:DigitalInformationShape> a sh:NodeShape ;
            sh:targetClass deix:Digital_Information ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Digital Information must have a label."
            ] ;
            sh:property [
                sh:path deix:informationValue ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "Digital Information must have a value."
            ] ;
            sh:property [
                sh:path deix:informationUnit ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "Digital Information should declare units."
            ] ;
            sh:property [
                sh:path deix:fromSource ;
                sh:minCount 1 ;
                sh:severity sh:Warning ;
                sh:message "Digital Information should declare its source tool."
            ] .
    """,
)

print(deix)
print(f"\n  Interior: EMPTY (will be populated by portal projections)")
print(f"  Boundary: SHACL shapes using deix: vocabulary")

Holon('DEIX Canonical', depth=0, interior=7t, boundary=38t, projection=0t, context=0t)

  Interior: EMPTY (will be populated by portal projections)
  Boundary: SHACL shapes using deix: vocabulary


## Step 3: Discover the DEIX Surface

Before building the portal, the SysML tool queries the DEIX boundary
shapes to discover what DEIX accepts.  This is the self-describing
surface pattern.

In [6]:
print(describe_surface(deix))

shapes = discover_target_shape(deix)
print("\n  Structured shape discovery:")
for cls, props in shapes.items():
    cls_short = cls.split("#")[-1] if "#" in cls else cls.split(":")[-1]
    print(f"\n  Class: {cls_short}")
    for p in props:
        req = "REQUIRED" if p.is_required else "optional"
        print(f"    {p.path_local:<25s} {req:<10s} {p.severity}")

Surface of 'DEIX Canonical':

  Target class: DEIX_Ontology#Digital_Artifact (https://semantic.incose.org/DEIX_Ontology#Digital_Artifact)
    label                     REQUIRED   [string]
      └─ Digital Artifact must have a label (DEIX).
    hasParameter              REQUIRED   (Warning)
      └─ Digital Artifact should declare parameters.

  Target class: DEIX_Ontology#Digital_Information (https://semantic.incose.org/DEIX_Ontology#Digital_Information)
    label                     REQUIRED   [string]
      └─ Digital Information must have a label.
    fromSource                REQUIRED   (Warning)
      └─ Digital Information should declare its source tool.
    informationUnit           REQUIRED   [string] (Warning)
      └─ Digital Information should declare units.
    informationValue          REQUIRED  
      └─ Digital Information must have a value.

  Structured shape discovery:

  Class: Digital_Artifact
    label                     REQUIRED   Violation
    hasParameter      

## Step 4: Build the SysML → DEIX Portal

The SPARQL CONSTRUCT translates `sysml:` vocabulary into `deix:` vocabulary.
This is the mapping layer — it encodes the domain knowledge of how
SysML model elements correspond to DEIX Digital Artifact concepts.

In [8]:
SYSML_TO_DEIX = """
PREFIX sysml: <urn:sysml:>
PREFIX deix:  <https://semantic.incose.org/DEIX_Ontology#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?block a deix:Digital_Artifact .
    ?block rdfs:label ?name .
    ?block deix:hasParameter ?param .

    ?param a deix:Digital_Information .
    ?param rdfs:label ?pname .
    ?param deix:informationValue ?pval .
    ?param deix:informationUnit ?punit .
    ?param deix:fromSource "SysML v2 Modeler" .
    ?param deix:belongsTo ?block .
}
WHERE {
    ?block a sysml:Block .
    ?block sysml:name ?name .

    ?param a sysml:ValueProperty .
    ?param sysml:owner ?block .
    ?param sysml:name ?pname .
    ?param sysml:value ?pval .
    OPTIONAL { ?param sysml:unit ?punit }
}
"""

portal_sysml_deix = TransformPortal(
    iri="urn:portal:sysml-to-deix",
    source=sysml,
    target=deix,
    label="SysML → DEIX (canonical lift)",
    construct_query=SYSML_TO_DEIX,
)

print(f"  Created: {portal_sysml_deix}")
print(f"  Portal registered in SysML boundary: {len(sysml.boundary)} triples")
print(f"\n  The portal's CONSTRUCT translates:")
print(f"    sysml:Block          →  deix:Digital_Artifact")
print(f"    sysml:ValueProperty  →  deix:Digital_Information")
print(f"    sysml:name           →  rdfs:label")
print(f"    sysml:value          →  deix:informationValue")
print(f"    sysml:unit           →  deix:informationUnit")

  Created: TransformPortal('SysML → DEIX (canonical lift)', 'SysML v2 Modeler' → 'DEIX Canonical', open, CONSTRUCT)
  Portal registered in SysML boundary: 21 triples

  The portal's CONSTRUCT translates:
    sysml:Block          →  deix:Digital_Artifact
    sysml:ValueProperty  →  deix:Digital_Information
    sysml:name           →  rdfs:label
    sysml:value          →  deix:informationValue
    sysml:unit           →  deix:informationUnit


## Step 5: Traverse the Portal — SysML Interior → DEIX Form

The CONSTRUCT executes against the SysML interior and produces
triples in DEIX vocabulary.  These triples go into the DEIX holon's
interior — making them the canonical representation.

In [10]:
deix_projection = portal_sysml_deix.traverse(sysml.interior)

print(f"  Source: {sysml.interior.identifier} ({len(sysml.interior)} triples, sysml: vocab)")
print(f"  Output: {len(deix_projection)} triples in deix: vocabulary\n")

print("  Projected triples (DEIX canonical form):\n")
for s, p, o in sorted(deix_projection):
    s_short = str(s).split(":")[-1]
    p_short = (str(p)
               .replace("https://semantic.incose.org/DEIX_Ontology#", "deix:")
               .replace("http://www.w3.org/2000/01/rdf-schema#", "rdfs:")
               .replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:"))
    print(f"    {s_short:30s} {p_short:30s} {o.n3()[:40]}")

# Load into DEIX interior — this IS the canonical representation now
for t in deix_projection:
    deix.interior.add(t)

print(f"\n  DEIX interior now has {len(deix.interior)} triples")

  Source: urn:holon:tool:sysml/interior (26 triples, sysml: vocab)
  Output: 23 triples in deix: vocabulary

  Projected triples (DEIX canonical form):

    tms                            rdf:type                       <https://semantic.incose.org/DEIX_Ontolo
    tms                            rdfs:label                     "ThermalMgmtSubsystem"
    tms                            deix:hasParameter              <urn:sysml:param:tms:flow>
    tms                            deix:hasParameter              <urn:sysml:param:tms:mass>
    tms                            deix:hasParameter              <urn:sysml:param:tms:power>
    flow                           rdf:type                       <https://semantic.incose.org/DEIX_Ontolo
    flow                           rdfs:label                     "coolantFlowRate"
    flow                           deix:belongsTo                 <urn:sysml:block:tms>
    flow                           deix:fromSource                "SysML v2 Modeler"
    flo

## Step 6: Record Provenance — The Hyperedge in Action

The provenance activity references BOTH graph IRIs: it `prov:used`
the SysML interior graph and `prov:generated` content in the DEIX
interior graph.  The activity IRI appears in the SysML context graph.

**This is the hyperedge**: the context graph (a named graph) contains
triples whose subjects and objects are OTHER named graph IRIs.

In [12]:
prov = ProvenanceTracker(
    agent_iri="urn:agent:deix-translator",
    agent_label="DEIX Translation Pipeline",
)

activity_id = prov.record_traversal(
    portal_iri=portal_sysml_deix.iri,
    source=sysml,
    target=deix,
    notes="Translated SysML v2 model data to DEIX canonical form.",
)

print(f"  Activity: {activity_id}")
print(f"\n  Hyperedge pattern — the context graph references other graph IRIs:\n")

for s, p, o in sorted(sysml.context):
    s_str = str(s)
    o_str = str(o)
    # Highlight cross-graph references
    is_cross = ("interior" in s_str or "interior" in o_str or
                "projection" in o_str or "portal" in o_str)
    marker = " ◄── cross-graph ref" if is_cross else ""
    s_short = s_str.rsplit("/", 1)[-1].rsplit(":", 1)[-1][:35]
    p_short = str(p).split("#")[-1].split("/")[-1][:25]
    o_short = o_str[:45]
    print(f"    {s_short:37s} {p_short:27s} {o_short}{marker}")

  Activity: urn:prov:traversal:9db77286dc60

  Hyperedge pattern — the context graph references other graph IRIs:

    sysml                                 urn:cga:memberOf            urn:holon:domain:engineering


## Step 7: Build the Simulation Tool Holon

The simulation tool has its OWN local vocabulary (`sim:`).
Its boundary shapes define what it accepts as input.

In [13]:
sim = Holon(
    iri="urn:holon:tool:sim",
    label="Thermal Simulation",
    depth=1,

    # Interior starts empty — will be populated by DEIX→Sim portal
    interior_ttl="",

    # Boundary: what the sim accepts (sim: vocabulary)
    boundary_ttl="""
        @prefix sim: <urn:sim:> .

        <urn:shapes:sim:InputBlockShape> a sh:NodeShape ;
            sh:targetClass sim:InputBlock ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "InputBlock must have a label."
            ] ;
            sh:property [
                sh:path sim:hasInput ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "InputBlock must have at least one input parameter."
            ] .

        <urn:shapes:sim:InputParamShape> a sh:NodeShape ;
            sh:targetClass sim:InputParam ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "InputParam must have a label."
            ] ;
            sh:property [
                sh:path sim:numericValue ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "InputParam must have a numeric value."
            ] ;
            sh:property [
                sh:path sim:unitOfMeasure ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "InputParam should declare units."
            ] .
    """,

    context_ttl="""
        <urn:holon:tool:sim> cga:memberOf <urn:holon:domain:engineering> .
    """,
)

print(sim)
print(f"\n  Sim surface (what it accepts):\n")
print(describe_surface(sim))

Holon('Thermal Simulation', depth=1, interior=7t, boundary=32t, projection=0t, context=1t)

  Sim surface (what it accepts):

Surface of 'Thermal Simulation':

  Target class: InputBlock (urn:sim:InputBlock)
    label                     REQUIRED   [string]
      └─ InputBlock must have a label.
    hasInput                  REQUIRED  
      └─ InputBlock must have at least one input parameter.

  Target class: InputParam (urn:sim:InputParam)
    label                     REQUIRED   [string]
      └─ InputParam must have a label.
    numericValue              REQUIRED  
      └─ InputParam must have a numeric value.
    unitOfMeasure             REQUIRED   [string] (Warning)
      └─ InputParam should declare units.


## Step 8: Build the DEIX → Sim Portal

Now the second translation: from DEIX canonical vocabulary to the
simulation tool's local vocabulary.

In [15]:
DEIX_TO_SIM = """
PREFIX deix: <https://semantic.incose.org/DEIX_Ontology#>
PREFIX sim:  <urn:sim:>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?block a sim:InputBlock .
    ?block rdfs:label ?name .
    ?block sim:hasInput ?param .

    ?param a sim:InputParam .
    ?param rdfs:label ?pname .
    ?param sim:numericValue ?pval .
    ?param sim:unitOfMeasure ?punit .
}
WHERE {
    ?block a deix:Digital_Artifact .
    ?block rdfs:label ?name .
    ?block deix:hasParameter ?param .

    ?param a deix:Digital_Information .
    ?param rdfs:label ?pname .
    ?param deix:informationValue ?pval .
    OPTIONAL { ?param deix:informationUnit ?punit }
}
"""

portal_deix_sim = TransformPortal(
    iri="urn:portal:deix-to-sim",
    source=deix,
    target=sim,
    label="DEIX → Sim (canonical lower)",
    construct_query=DEIX_TO_SIM,
)

print(f"  Created: {portal_deix_sim}")
print(f"\n  Translation:")
print(f"    deix:Digital_Artifact    →  sim:InputBlock")
print(f"    deix:Digital_Information →  sim:InputParam")
print(f"    deix:informationValue    →  sim:numericValue")
print(f"    deix:informationUnit     →  sim:unitOfMeasure")

  Created: TransformPortal('DEIX → Sim (canonical lower)', 'DEIX Canonical' → 'Thermal Simulation', open, CONSTRUCT)

  Translation:
    deix:Digital_Artifact    →  sim:InputBlock
    deix:Digital_Information →  sim:InputParam
    deix:informationValue    →  sim:numericValue
    deix:informationUnit     →  sim:unitOfMeasure


## Step 9: Traverse DEIX → Sim and Validate

The second portal translates DEIX canonical data into the sim's
local vocabulary.  Validation confirms the sim's boundary accepts it.

In [17]:
sim_input = portal_deix_sim.traverse(deix.interior)

print(f"  Source: DEIX interior ({len(deix.interior)} triples, deix: vocab)")
print(f"  Output: {len(sim_input)} triples in sim: vocabulary\n")

print("  Projected triples (simulation-ready):\n")
for s, p, o in sorted(sim_input):
    s_short = str(s).split(":")[-1]
    p_short = (str(p)
               .replace("urn:sim:", "sim:")
               .replace("http://www.w3.org/2000/01/rdf-schema#", "rdfs:")
               .replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:"))
    print(f"    {s_short:30s} {p_short:22s} {o.n3()[:40]}")

# Load into sim interior
for t in sim_input:
    sim.interior.add(t)

# Validate
result = validate_membrane(sim)
print(f"\n  Sim membrane validation: {result.health.value.upper()}")
if not result.conforms:
    for v in result.violations:
        print(f"    ✗ {v}")
for w in result.warnings:
    print(f"    ⚠ {w}")

# Record provenance for this leg too
prov.record_traversal(
    portal_iri=portal_deix_sim.iri,
    source=deix,
    target=sim,
    notes="Translated DEIX canonical data to simulation input format.",
)

  Source: DEIX interior (30 triples, deix: vocab)
  Output: 17 triples in sim: vocabulary

  Projected triples (simulation-ready):

    tms                            rdf:type               <urn:sim:InputBlock>
    tms                            rdfs:label             "ThermalMgmtSubsystem"
    tms                            sim:hasInput           <urn:sysml:param:tms:flow>
    tms                            sim:hasInput           <urn:sysml:param:tms:mass>
    tms                            sim:hasInput           <urn:sysml:param:tms:power>
    flow                           rdf:type               <urn:sim:InputParam>
    flow                           rdfs:label             "coolantFlowRate"
    flow                           sim:numericValue       "15.0"^^<http://www.w3.org/2001/XMLSchem
    flow                           sim:unitOfMeasure      "L/min"
    mass                           rdf:type               <urn:sim:InputParam>
    mass                           rdfs:label        

'urn:prov:traversal:9475d6b9cb5f'

## The Complete Pipeline, Visualized

```
SysML Interior          DEIX Interior            Sim Interior
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│ sysml:Block  │        │ deix:Digital  │        │ sim:Input    │
│ sysml:Value  │──────→ │   Artifact   │──────→ │   Block      │
│ Property     │ PORTAL │ deix:Digital  │ PORTAL │ sim:Input    │
│ (142.3 kg)   │ sysml→ │ Information  │ deix→  │   Param      │
│              │ deix   │ (142.3 kg)   │ sim    │ (142.3 kg)   │
└──────────────┘ CONSTR └──────────────┘ CONSTR └──────────────┘
       │          UCT          │          UCT          │
       │                       │                       │
  SysML Context          DEIX Context            Sim Context
  ┌────────────┐        ┌────────────┐        ┌────────────┐
  │ prov:used   │        │ prov:was   │        │ prov:was   │
  │  <.../      │        │ DerivedFrom│        │ DerivedFrom│
  │  interior>  │        │  <sysml/   │        │  <deix/    │
  │ prov:gen    │        │  interior> │        │  interior> │
  │  <deix/     │        │            │        │            │
  │  interior>  │        │            │        │            │
  └────────────┘        └────────────┘        └────────────┘

Context graphs reference interior graph IRIs → HYPEREDGES
```

## Step 10: Assemble the Holarchy and Show the Thread Structure

In [18]:
holarchy = Holarchy(label="DEIX Translation Holarchy")
holarchy.register(sysml)
holarchy.register(deix)
holarchy.register(sim)
holarchy.add_portal(portal_sysml_deix)
holarchy.add_portal(portal_deix_sim)

print(holarchy.summary())

# Show the hyperedge structure: which graph IRIs are referenced where
print(f"\n  Hyperedge inventory (graph IRIs referenced across layers):\n")

all_graph_iris = set()
for h in holarchy.holons:
    for layer in ["interior", "boundary", "projection", "context"]:
        g = getattr(h, layer)
        all_graph_iris.add(str(g.identifier))

# Find cross-references
for h in holarchy.holons:
    for layer in ["interior", "boundary", "projection", "context"]:
        g = getattr(h, layer)
        for s, p, o in g:
            for term in [s, o]:
                term_str = str(term)
                if term_str in all_graph_iris and term_str != str(g.identifier):
                    g_short = str(g.identifier).split(":")[-1]
                    t_short = term_str.split(":")[-1]
                    p_short = str(p).split("#")[-1].split("/")[-1]
                    print(f"    {g_short:35s} references {t_short:35s} via {p_short}")

Holarchy: DEIX Translation Holarchy
  Holon('SysML v2 Modeler', depth=1, interior=26t, boundary=21t, projection=0t, context=1t)
  Holon('DEIX Canonical', depth=0, interior=30t, boundary=46t, projection=0t, context=13t)
  Holon('Thermal Simulation', depth=1, interior=24t, boundary=32t, projection=0t, context=14t)
  Portals:
    TransformPortal('SysML → DEIX (canonical lift)', 'SysML v2 Modeler' → 'DEIX Canonical', open, CONSTRUCT)
    TransformPortal('DEIX → Sim (canonical lower)', 'DEIX Canonical' → 'Thermal Simulation', open, CONSTRUCT)

  Hyperedge inventory (graph IRIs referenced across layers):

    sysml/interior                      references sysml/context                       via urn:cga:contextGraph
    sysml/interior                      references sysml/boundary                      via urn:cga:boundaryGraph
    sysml/interior                      references sysml/projection                    via urn:cga:projectionGraph
    canonical/interior                  references ca

## What This Demonstrates

1. **Three local vocabularies** (`sysml:`, `deix:`, `sim:`) — each tool
   keeps its own representation in its own interior.

2. **DEIX as canonical hub** — data enters DEIX via one portal and
   exits via another.  Adding a new tool means adding two portals, not
   N converters.

3. **Self-describing surfaces** — the DEIX boundary SHACL shapes are
   queryable by any tool to discover what DEIX accepts.

4. **Named graphs as hyperedges** — context graphs reference interior
   and projection graph IRIs, creating cross-layer structure.

5. **Provenance at every portal** — every traversal is a `prov:Activity`
   that `prov:used` the source interior and `prov:generated` the target
   interior contribution.

6. **Validation at every boundary** — SHACL shapes in the target
   boundary validate the projected output before it enters.